# Phase 2: Baseline Models

| | |
|---|---|
| **Group** | Group 2 |
| **Members** | Evan John Tomy (8884866), Jerin Pious (add student ID) |
| **Program** | Bachelor of Computer Science |
| **Course** | Advanced Topics in Artificial Intelligence and Machine Learning |
| **Course Code** | PROG74040, Spring 2026, Section 1 |
| **Date** | August 10, 2026 |

---

**Notebook 3 of 5** reads `train_clean.csv` / `val_clean.csv` / `pos_weight.pt` from Notebook 2 and writes `baseline_metrics.csv` and `lstm_model.pt`, used by Notebook 5's final comparison.

## Purpose
Before justifying the cost and complexity of fine-tuning a transformer, this notebook establishes a performance floor with two simple, well-understood models: TF-IDF + Logistic Regression, and an LSTM using pretrained GloVe word embeddings. Both are committed to in the Phase 1 proposal specifically as baselines to measure the advanced model's improvement against.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

torch.manual_seed(42)
np.random.seed(42)

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

train = pd.read_csv("train_clean.csv")
val = pd.read_csv("val_clean.csv")
print("Train:", train.shape, "Val:", val.shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

y_train = train[LABEL_COLS].values
y_val = val[LABEL_COLS].values

Train: (135614, 9) Val: (23932, 9)
Device: cuda


In [2]:
def compute_metrics(y_true, y_prob, y_pred, label_cols, model_name):
    rows = []
    for i, label in enumerate(label_cols):
        rows.append({
            "model": model_name,
            "label": label,
            "roc_auc": roc_auc_score(y_true[:, i], y_prob[:, i]),
            "f1": f1_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "precision": precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "recall": recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
        })
    df = pd.DataFrame(rows)
    macro = df[["roc_auc", "f1", "precision", "recall"]].mean()
    print(f"=== {model_name} — per-label ===")
    print(df.set_index("label").round(4))
    print(f"\n=== {model_name} — macro-averaged ===")
    print(macro.round(4))
    return df

print("compute_metrics() ready — shared by every model in this project for a fair comparison")

compute_metrics() ready — shared by every model in this project for a fair comparison


## Baseline 1: TF-IDF + Logistic Regression

One independent binary classifier per label (`OneVsRestClassifier`), `class_weight="balanced"` to handle the imbalance, the same independent-per-label framing used everywhere else in this project.

In [3]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(train["clean_text"])
X_val_tfidf = tfidf.transform(val["clean_text"])
print("TF-IDF matrix:", X_train_tfidf.shape)

tfidf_lr = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"))
tfidf_lr.fit(X_train_tfidf, y_train)

tfidf_val_probs = tfidf_lr.predict_proba(X_val_tfidf)
tfidf_val_preds = (tfidf_val_probs >= 0.5).astype(int)
print("Trained. Val predictions shape:", tfidf_val_probs.shape)

TF-IDF matrix: (135614, 20000)


Trained. Val predictions shape: (23932, 6)


In [4]:
tfidf_lr_metrics = compute_metrics(y_val, tfidf_val_probs, tfidf_val_preds, LABEL_COLS, "TF-IDF+LogReg")

=== TF-IDF+LogReg — per-label ===
                       model  roc_auc      f1  precision  recall
label                                                           
toxic          TF-IDF+LogReg   0.9703  0.7174     0.6130  0.8648
severe_toxic   TF-IDF+LogReg   0.9804  0.4039     0.2634  0.8661
obscene        TF-IDF+LogReg   0.9804  0.7402     0.6374  0.8824
threat         TF-IDF+LogReg   0.9942  0.3800     0.2500  0.7917
insult         TF-IDF+LogReg   0.9773  0.6520     0.5188  0.8772
identity_hate  TF-IDF+LogReg   0.9701  0.3234     0.2045  0.7725

=== TF-IDF+LogReg — macro-averaged ===
roc_auc      0.9788
f1           0.5361
precision    0.4145
recall       0.8425
dtype: float64


**Reading these numbers**: ROC-AUC is strong across every label (0.97-0.99), meaning the model ranks toxic vs. non-toxic comments well. F1 is much weaker for the rare labels (`threat` 0.38, `identity_hate` 0.32) specifically because this baseline uses a fixed, untuned 0.5 decision threshold, the same pattern later found and corrected for DistilBERT in Notebook 4. This is a fair baseline number precisely because no threshold tuning was applied to it either.

## Baseline 2: LSTM + GloVe (100d)

GloVe vectors downloaded directly from the Stanford NLP group (the source cited in the Phase 1 proposal, Pennington et al., 2014), not via a repackaged library, to avoid an extra compiled dependency. Runs on the local GPU: the proposal noted this baseline "requires no GPU," which just means it *can* run on CPU; running it on the 3060 instead is strictly faster with no downside.

In [5]:
import os
import zipfile

GLOVE_ZIP = "glove.6B.zip"
GLOVE_FILE = "glove.6B.100d.txt"
GLOVE_PATH = os.path.join("glove", GLOVE_FILE)

if not os.path.exists(GLOVE_PATH):
    os.makedirs("glove", exist_ok=True)
    print("Extracting 100d vectors from", GLOVE_ZIP)
    with zipfile.ZipFile(GLOVE_ZIP) as z:
        z.extract(GLOVE_FILE, "glove")

embeddings_index = {}
with open(GLOVE_PATH, encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        embeddings_index[parts[0]] = np.asarray(parts[1:], dtype="float32")

EMBED_DIM = len(next(iter(embeddings_index.values())))
print(f"Loaded {len(embeddings_index):,} GloVe vectors, dim={EMBED_DIM}")

Loaded 400,000 GloVe vectors, dim=100


In [6]:
from collections import Counter

MAX_LEN = 128
VOCAB_SIZE = 20000

def simple_tokenize(text):
    return text.lower().split()

counter = Counter()
for text in train["clean_text"]:
    counter.update(simple_tokenize(text))

vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

embedding_matrix = np.zeros((len(vocab), EMBED_DIM), dtype=np.float32)
hits = 0
for word, idx in vocab.items():
    vec = embeddings_index.get(word)
    if vec is not None:
        embedding_matrix[idx] = vec
        hits += 1
print(f"Vocab size: {len(vocab):,} | GloVe coverage: {hits:,} ({hits/len(vocab)*100:.1f}%)")

Vocab size: 20,000 | GloVe coverage: 12,503 (62.5%)


**A real limitation of this baseline**: only 62.5% of the LSTM's 20,000-word vocabulary has a matching pretrained GloVe vector: the other 37.5% (likely dominated by slang, misspellings, and leetspeak substitutions like "fck" or "u") start from a zero vector with no pretrained meaning at all. GloVe was trained on relatively clean Wikipedia/news text, which doesn't cover the informal, deliberately-obfuscated language that toxic comments often use. This is a plausible, concrete reason the LSTM underperforms TF-IDF later: TF-IDF has no such gap, since every word it sees gets its own learned weight directly.

In [7]:
from torch.utils.data import Dataset, DataLoader

def encode(text, vocab, max_len=MAX_LEN):
    tokens = simple_tokenize(text)[:max_len]
    ids = [vocab.get(t, vocab["<unk>"]) for t in tokens]
    ids += [vocab["<pad>"]] * (max_len - len(ids))
    return ids

class ToxicDataset(Dataset):
    def __init__(self, df, vocab):
        self.texts = df["clean_text"].tolist()
        self.labels = df[LABEL_COLS].values.astype(np.float32)
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx], self.vocab)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx])

train_loader = DataLoader(ToxicDataset(train, vocab), batch_size=64, shuffle=True)
val_loader = DataLoader(ToxicDataset(val, vocab), batch_size=64)
print("Batches — train:", len(train_loader), "val:", len(val_loader))

Batches — train: 2119 val: 374


In [8]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_labels=6):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=False, padding_idx=0
        )
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, x):
        emb = self.embedding(x)
        _, (h, _) = self.lstm(emb)
        pooled = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.dropout(pooled))

lstm_model = LSTMClassifier(embedding_matrix).to(device)
pos_weight_tensor = torch.load("pos_weight.pt").to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)
print(lstm_model)

LSTMClassifier(
  (embedding): Embedding(20000, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=6, bias=True)
)


In [9]:
EPOCHS = 3
for epoch in range(EPOCHS):
    lstm_model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = lstm_model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} — train loss: {total_loss/len(train_loader):.4f}")

Epoch 1/3 — train loss: 0.6265


Epoch 2/3 — train loss: 0.3748


Epoch 3/3 — train loss: 0.2984


In [10]:
lstm_model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for x, y in val_loader:
        logits = lstm_model(x.to(device))
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(y.numpy())

lstm_val_probs = np.concatenate(all_probs)
lstm_val_labels = np.concatenate(all_labels)
lstm_val_preds = (lstm_val_probs >= 0.5).astype(int)

lstm_metrics = compute_metrics(lstm_val_labels, lstm_val_probs, lstm_val_preds, LABEL_COLS, "LSTM+GloVe")

=== LSTM+GloVe — per-label ===
                    model  roc_auc      f1  precision  recall
label                                                        
toxic          LSTM+GloVe   0.9595  0.5150     0.3557  0.9324
severe_toxic   LSTM+GloVe   0.9881  0.1728     0.0947  0.9874
obscene        LSTM+GloVe   0.9758  0.4501     0.2950  0.9487
threat         LSTM+GloVe   0.9781  0.0401     0.0205  0.9861
insult         LSTM+GloVe   0.9701  0.3856     0.2413  0.9594
identity_hate  LSTM+GloVe   0.9660  0.1103     0.0585  0.9573

=== LSTM+GloVe — macro-averaged ===
roc_auc      0.9729
f1           0.2790
precision    0.1776
recall       0.9619
dtype: float64


**LSTM vs. TF-IDF, on validation**: macro F1 (0.279) is meaningfully worse than TF-IDF's 0.536, despite the LSTM being the more sophisticated model: it understands word order, TF-IDF doesn't. The GloVe coverage gap noted above is the most likely explanation: a model that can reason about sequence but has weak input representations for the exact words that signal toxicity will lose to a simpler model that represents every word directly, no matter how it processes them.

In [11]:
all_baseline_metrics = pd.concat([tfidf_lr_metrics, lstm_metrics], ignore_index=True)
all_baseline_metrics.to_csv("baseline_metrics.csv", index=False)
torch.save(lstm_model.state_dict(), "lstm_model.pt")
print("Saved baseline_metrics.csv")
print(all_baseline_metrics)

Saved baseline_metrics.csv
            model          label   roc_auc        f1  precision    recall
0   TF-IDF+LogReg          toxic  0.970278  0.717438   0.612983  0.864806
1   TF-IDF+LogReg   severe_toxic  0.980395  0.403902   0.263359  0.866109
2   TF-IDF+LogReg        obscene  0.980434  0.740152   0.637400  0.882399
3   TF-IDF+LogReg         threat  0.994195  0.380000   0.250000  0.791667
4   TF-IDF+LogReg         insult  0.977279  0.651982   0.518778  0.877223
5   TF-IDF+LogReg  identity_hate  0.970086  0.323413   0.204517  0.772512
6      LSTM+GloVe          toxic  0.959539  0.514995   0.355740  0.932403
7      LSTM+GloVe   severe_toxic  0.988059  0.172830   0.094703  0.987448
8      LSTM+GloVe        obscene  0.975791  0.450103   0.295042  0.948698
9      LSTM+GloVe         threat  0.978148  0.040079   0.020455  0.986111
10     LSTM+GloVe         insult  0.970106  0.385636   0.241321  0.959356
11     LSTM+GloVe  identity_hate  0.966028  0.110322   0.058534  0.957346


## Summary
Both baselines are trained, scored, and saved. TF-IDF+LogReg (macro F1 0.536) outperforms LSTM+GloVe (macro F1 0.279) on validation, a real result, not a coincidence, for the reasons noted above (GloVe's coverage gap). These numbers, and `lstm_model.pt`, feed directly into Notebook 5's final comparison against DistilBERT.